# Project data exploration and scatter plots - looking at relplot

**Group name:** _YOUR GROUP NAME_
**Created by:** _NAME 1 (SJSU ID), NAME 2 (SJSU ID), ..._
**Last updated:** _DATE_

**Dataset:** NBA Shot Logs, 2014-15 season (Kaggle, `shot_logs.csv`). One row per shot attempt, with shot distance,
closest-defender distance, shot clock, dribbles, touch time, and whether the shot was made.

### Import your data
Upload `shot_logs.csv` to the Colab session (folder icon on the left), **or** mount Google Drive and change the path below.

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_theme(style="whitegrid")

# Option A: file uploaded directly to the Colab session
PATH = "shot_logs.csv"

# Option B: Google Drive (uncomment)
# from google.colab import drive
# drive.mount('/content/drive')
# PATH = "/content/drive/MyDrive/YOUR_FOLDER/shot_logs.csv"

df = pd.read_csv(PATH)
print(df.shape)
df.head()

## Q1: Data background
*(Draft - edit so it is in your own words, and fill in anything marked CHECK after running the cells below.)*

**Source:** Kaggle "NBA Shot Logs" (originally built from NBA SportVU player-tracking data), covering the 2014-15 regular season.
**Size:** about 128,000 shot attempts (CHECK `df.shape` above) and 21 columns, one row per shot.
**Topic:** every shot is described by the shooter, the closest defender, how far the shot was, how far away the defender was,
the shot clock, dribbles and touch time before the shot, the period, and the result (made / missed).
**Motivation:** we want to understand what makes a shot more or less likely to go in, then train models that predict
make vs. miss from information available *before* the ball leaves the shooter's hand. Coaches and analysts care because
it separates good shot selection from good shooting.

## Q2: Show the number of missing data in each column of the dataframe
If you have missing data, how much? Which column(s)? Do you think it will impact your output?

In [ ]:
missing = df.isna().sum()
missing_df = pd.DataFrame({"missing": missing, "percent": (missing / len(df) * 100).round(2)})
missing_df.sort_values("missing", ascending=False)

In [ ]:
# Is SHOT_CLOCK missing at random? A common explanation is that the shot clock is turned off
# when fewer than 24 seconds remain in the period. Check it:
def to_seconds(t):
    m, s = str(t).split(":")
    return int(m) * 60 + int(s)

df["GAME_CLOCK_SEC"] = df["GAME_CLOCK"].apply(to_seconds)

no_clock = df[df["SHOT_CLOCK"].isna()]
print("Rows missing SHOT_CLOCK:", len(no_clock))
print("Share of those with <= 24 seconds left in the period:",
      round((no_clock["GAME_CLOCK_SEC"] <= 24).mean() * 100, 1), "%")

**Your answer:** _Write what you found (which column, how many rows, and the reason). Decide how you will handle it later
(drop those rows, fill with the game clock, or add a flag column) and say whether it could bias the results._

## Q3: Show the data type for each column of the dataframe
Is the data type as expected? Is it clear what the content/value of each column is?

In [ ]:
df.dtypes

In [ ]:
# Things worth checking: GAME_CLOCK is text, MATCHUP packs date + teams into one string,
# and TOUCH_TIME should never be negative.
print("Negative TOUCH_TIME rows:", (df["TOUCH_TIME"] < 0).sum())
print(df["SHOT_RESULT"].value_counts())
print(df["PTS_TYPE"].value_counts())
df[["TOUCH_TIME", "SHOT_DIST", "CLOSE_DEF_DIST", "DRIBBLES", "SHOT_CLOCK"]].describe().round(2)

**Your answer:** _Which types are as expected and which are not (e.g. GAME_CLOCK is a string, TOUCH_TIME has impossible
negative values). Say how you will clean them._

In [ ]:
# Light cleaning used in the plots below (keep the original df untouched)
clean = df[df["TOUCH_TIME"] >= 0].copy()
clean["SHOT_RESULT"] = clean["SHOT_RESULT"].astype("category")
clean["FGM"] = clean["FGM"].astype(int)          # 1 = made, 0 = missed
print("Rows kept:", len(clean), "of", len(df))

## Q4: Your own question using relplot()
**Question:** How does field-goal percentage change as shot distance increases, and is the pattern different for 2-point and 3-point attempts?

In [ ]:
clean["DIST_FT"] = clean["SHOT_DIST"].round().astype(int)

g = sns.relplot(
    data=clean[clean["DIST_FT"] <= 35],
    x="DIST_FT", y="FGM", hue="PTS_TYPE", kind="line",
    errorbar=None, height=5, aspect=1.5, marker="o", palette="deep",
)
g.set_axis_labels("Shot distance (feet)", "Field-goal percentage (made = 1)")
g.figure.suptitle("Field-goal percentage by shot distance", y=1.03)
plt.show()

In [ ]:
# Second view: does defender distance matter? (sample so the scatter is readable)
sample = clean.sample(4000, random_state=42)
g = sns.relplot(
    data=sample, x="SHOT_DIST", y="CLOSE_DEF_DIST", hue="SHOT_RESULT",
    alpha=0.5, height=5, aspect=1.4,
)
g.set_axis_labels("Shot distance (feet)", "Closest defender distance (feet)")
g.figure.suptitle("Shot distance vs. defender distance (4,000-shot sample)", y=1.03)
plt.show()

**Your answer:** _What relationship do you see? Is it what you expected?_

## Q5: Your own question using catplot()
**Question:** Are players more accurate on catch-and-shoot attempts (few dribbles) than after several dribbles?

In [ ]:
clean["DRIBBLE_GROUP"] = pd.cut(
    clean["DRIBBLES"], bins=[-1, 0, 2, 6, 100], labels=["0", "1-2", "3-6", "7+"]
)

g = sns.catplot(
    data=clean, x="DRIBBLE_GROUP", y="FGM", hue="PTS_TYPE", kind="bar",
    errorbar=None, height=5, aspect=1.4,
)
g.set_axis_labels("Dribbles before the shot", "Field-goal percentage")
g.figure.suptitle("Field-goal percentage by number of dribbles", y=1.03)
plt.show()

**Your answer:** _What do you notice? Keep in mind that shots after many dribbles may also be farther or more contested._

## Q6: Create a plot with small multiples plots
Two features (shot distance and closest-defender distance) with the game period as the single category.
3 plots per row, height = 5, point size = 20.

In [ ]:
sample = clean.sample(6000, random_state=42)

g = sns.relplot(
    data=sample, x="SHOT_DIST", y="CLOSE_DEF_DIST",
    col="PERIOD", col_wrap=3, height=5, s=20,
)
g.set_axis_labels("Shot distance (feet)", "Closest defender distance (feet)")
g.set_titles("Period {col_name}")
g.figure.suptitle("Shot distance vs. defender distance in each period (periods 5-7 are overtime)", y=1.02)
plt.show()

**Your answer:** _Does the relationship look the same in every period? (Overtime panels have far fewer shots.)_